# Task 3 — Online estimation of the tolerance $c$

## The experiment
The dataset comes from `CODE/LF_DATA/LDF.m`, which builds the **least favourable model** of the
ambiguity set $\mathcal{B}_{c,k}$ ([UAVF] Sec. IV) using a tolerance $c(t)$ that is stored inside
the `.mat` file. The dataset therefore has a **known ground truth for $c$**, and the question the
three estimators answer is: *how well is that tolerance recovered and exploited online?*

| Estimator | Tolerance | 
|---|---|
| **REKF** (baseline) | one **constant** $c^*$, tuned on the training set over a grid ([UAVF] Sec. IV) |
| **Original RT-KalmanNet** | MLP with output feedback, estimates $c_t$ online |
| **Proposed RT-KalmanNet** | GRU, estimates $c_t$ online |

All three are scored with the **same metric** — the posterior state MSE over the test
trajectories — so the numbers are directly comparable.

## One dataset per run
`LF_DATA/` ships several $c(t)$ shapes (constant, linear, ...). The notebook analyses **one** of
them per run: point `DATA_FILE` at the `.mat` file in Section 2 and run top to bottom. Results,
checkpoints and history are filed under `Results_task3/<shape>/`, so running it again on another
shape neither overwrites the previous run nor lets its model be reused. Section 17 then collects
whichever shapes have been run into one comparative table.

## 1. Imports and project bootstrap

Locates the project root by walking up from the current directory until a `Simulations/` folder is
found, adds it to `sys.path`, and `chdir`s into it — so every relative import and file path below
resolves correctly regardless of where the notebook is launched from.

In [2]:
import os, sys, math, time, random, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import scipy.io as sio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

# find the project root (wherever Simulations/ lives) so paths work no matter
# where this notebook is opened from
_here = Path.cwd()
root = next((q for q in [_here, *_here.parents] if (q / "Simulations").is_dir()), _here)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.chdir(root)

from Simulations import config
from Simulations.Extended_sysmdl import SystemModel
from RobustKalmanPY.robust_kalman_original import RobustKalman as Orig_RTKnet   # MLP, professor's
from RobustKalmanPY.robust_kalman_proposed import RobustKalman as Prop_RTKnet   # GRU, ours
from Pipelines.Pipeline_REKF import Pipeline_REKF                       # RT-KalmanNet pipeline

crit = nn.MSELoss(reduction="mean")

def to_dB(x):
    """Linear MSE -> dB."""
    return 10.0 * math.log10(float(x))

print("project root :", root)
print("torch        :", torch.__version__)

project root : c:\Users\Andrea\Documents\Università\Magistrale\Learning dynamical systems\RT-KalmanNet-project2026\CODE\RT_KFNET
torch        : 2.13.0+cpu


## 2. Configuration (Edit only this cell)

This cell centralizes all experimental parameters: evaluated estimators, dataset splits, sizing, and hyperparameters for the system model and training pipeline.

**Choosing the experiment — `DATA_FILE`.** The notebook analyses **one** dataset per run. To study a
different $c(t)$ shape, point `DATA_FILE` at another `LF_DATA/*.mat` and re-run top to bottom.

| | path |
|---|---|
| checkpoints, tables, history | `Results_task3/<kind>/` |

So the runs never overwrite each other, and a model trained on one shape can never be loaded to
test another. If a file does not follow the naming convention the cell stops with an explicit
error rather than guessing; set `DATASET_KIND` by hand in that case.

**Model selection — `RETRAIN`.**

* `RETRAIN = True` — train both networks, then save the checkpoints **and** the
  training/validation history.
* `RETRAIN = False` (**default**) — load the saved checkpoints **and** the saved
  history; nothing is trained. Section 9 still draws the loss curves and Sections 10–15 run
  normally. A missing checkpoint stops the notebook with an explicit message: it never trains
  silently.

**Computational Cost Note:**
The REKF procedure requires a $\theta$ bisection at each time step. Consequently, the trajectory length ($T$) strictly dictates the overall computational load. Training introduces additional significant overhead due to the backpropagation-through-time (BPTT) algorithm.

> *Recommendation:* Utilize `T_USE` and `QUICK_RUN` to perform a preliminary end-to-end validation run. This verifies correct system initialization without incurring the full computational cost.

In [3]:
# User Configuration
SEED        = 0
USE_CUDA    = False          # in practice the linear-algebra ops here are CPU-only anyway
DATA_FILE   = root.parent / "LF_DATA" / "data_constant_01.mat"
CKPT_ROOT   = "Results_task3"   # one sub-folder per dataset kind -> separate checkpoints

RETRAIN     = False          # True: train and save checkpoints + history; False: load both

# Estimators to run
INCLUDE_REKF     = True      # constant-tolerance baseline (grid search, no training)
INCLUDE_ORIGINAL = True      # professor's MLP
INCLUDE_PROPOSED = True      # our GRU

# Dataset parameters
TRAIN_RATIO, CV_RATIO, TEST_RATIO = 0.6, 0.15, 0.25
T_USE       = None           # None = full trajectory length; an int truncates it (fast runs)

# Experiment sizes
N_TRAIN_SEQ = None           # None = all training sequences, an int subsamples them
N_CV_SEQ    = 4              # validation sequences scored at every epoch
N_TEST      = 10             # test trajectories used for the final metrics
N_C_SEQ     = 6              # training sequences used for the REKF tolerance grid search
C_GRID      = np.geomspace(1e-4, 1.0, 20)   # candidate constant tolerances for the REKF

# Model hyperparameters
PROP_MODEL = {"input_feat_mode": 3, "gru_hidden_size": 128}
ORIG_MODEL = {"input_feat_mode": 3, "hidden_layers": [20, 20, 20, 20, 20]}

# Training hyperparameters
# Explicitly defining BPTT and clipping parameters here, as Pipeline_REKF 
# strictly requires them via `args` and they lack defaults in config.py.
PROP_TRAIN = {
    "n_steps": 10,           # total training epochs
    "n_batch": 8,
    "lr": 1e-3,
    "wd": 1e-4,
    "grad_clip": 1.0,        # max grad norm (None disables)
    "bptt_truncation": 50,   # window size for truncated BPTT (Proposed model only)
    "bptt_warmup_frac": 0.7, # epoch fraction using truncated BPTT before full BPTT
}
ORIG_TRAIN = {**PROP_TRAIN, "lr": 1e-3, "wd": 1e-3, "bptt_truncation": None, "bptt_warmup_frac": None}

# Preliminary validation
QUICK_RUN = False            # set to True for a fast run to check the code works 
if QUICK_RUN:
    T_USE, N_TRAIN_SEQ, N_CV_SEQ, N_TEST, N_C_SEQ = 60, 4, 2, 2, 2
    C_GRID = np.linspace(1e-3, 1.0, 3)
    PROP_TRAIN = {**PROP_TRAIN, "n_steps": 2, "n_batch": 2}
    ORIG_TRAIN = {**ORIG_TRAIN, "n_steps": 2, "n_batch": 2}

USE_CUDA = USE_CUDA and torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

# The c(t) shape, read off the filename: LFM datasets are named data_<kind>_<params>.mat, so
# the second token is the kind. Validated against the shapes LF_DATA/ can generate, because a
# hand-renamed file would otherwise silently send its checkpoints to the wrong folder. Set
# DATASET_KIND by hand just above this block to override the convention.
KNOWN_KINDS = ("constant", "linear")   # cf. LF_DATA/generate_constrained_*.m
if "DATASET_KIND" not in dir():
    DATASET_KIND = DATA_FILE.stem.split("_")[1] if "_" in DATA_FILE.stem else ""
if DATASET_KIND not in KNOWN_KINDS:
    raise ValueError(
        f"cannot tell the c(t) shape of '{DATA_FILE.name}': got {DATASET_KIND!r}, expected one "
        f"of {KNOWN_KINDS}.\nEither restore the data_<kind>_<params>.mat naming or set "
        f"DATASET_KIND explicitly in this cell.")

CKPT_DIR = os.path.join(CKPT_ROOT, DATASET_KIND)   # per-kind checkpoints, tables and history
PATH_RESULTS = CKPT_DIR + "/"
HIST_PATH = os.path.join(PATH_RESULTS, "training_history.json")
os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f"device = {DEVICE} | QUICK_RUN = {QUICK_RUN}")
print(f"dataset     : {DATA_FILE.name}   (c(t) shape: {DATASET_KIND})")
print(f"results     : {CKPT_DIR}/")
print("mode        : " + ("TRAIN FROM SCRATCH, saving checkpoints + history" if RETRAIN
                          else "LOAD saved checkpoints + history, nothing is trained"))

device = cpu | QUICK_RUN = False
dataset     : data_constant_01.mat   (c(t) shape: constant)
results     : Results_task3\constant/
mode        : LOAD saved checkpoints + history, nothing is trained


## 3. Dataset (Load, Align, Split)

The data required for tracking is loaded from the `.mat` file generated via the Matlab implementation (`LDF.m`).

* `yw` : The least-favorable measurement sequences, forming a tensor of shape `[M, n, T]`.
* `xw` : The corresponding least-favorable latent state sequences, forming a tensor of shape `[M, m, T+1]`.
* `c` : The ground-truth tolerance parameter characterizing the ambiguity set used during data generation.

**Target alignment for `Pipeline_REKF`:** 
The pipeline requires strict temporal alignment between inputs and targets: column *i* of the target tensor must represent the latent state corresponding to the observation in column *i* of the input tensor. Because the `LFM_data.m` script records states from $x_0$ to $x_T$ but observations only from $y_0$ to $y_{T-1}$, the trailing unmeasured state column in `xw` is truncated upon loading.

> **Alignment of the ground-truth sequence $c(t)$:** 
> The generation script `LDF.m` computes the tolerance sequence `c` over the full operational horizon $T$. However, the subsequent sampling mechanism in `LFM_data.m` outputs roughly $\text{round}(T/2)$ samples. Consequently, the saved vector `c` contains more entries than the actual processed data length. We align this by extracting the first $T$ samples (`c_true = c[:T]`), which accurately reflect the tolerance experienced by the evaluated trajectories.

In [4]:
# Data Loading and Tensor Initialization
mat = sio.loadmat(DATA_FILE)

xw = torch.tensor(mat["xw"], dtype=torch.float32).permute(2, 0, 1).to(DEVICE)   # latent state sequences [M, n, T+1]
yw = torch.tensor(mat["yw"], dtype=torch.float32).permute(2, 0, 1).to(DEVICE)   # observation sequences [M, m, T]

A = torch.tensor(mat["A"], dtype=torch.float32).to(DEVICE)
C = torch.tensor(mat["C"], dtype=torch.float32).to(DEVICE)
Q = torch.tensor(mat["Q"], dtype=torch.float32).to(DEVICE)
R = torch.tensor(mat["R"], dtype=torch.float32).to(DEVICE)

m = A.shape[0]        # latent state vector dimension
n = C.shape[0]        # observation vector dimension

# System Dynamics Definition
def f(x):              # state-evolution function
    return A @ x

def h(x):               # observation (emission) mapping
    return C @ x

#  Trajectory Truncation and Formatting 
T = yw.shape[-1]
if T_USE is not None:             # enforce trajectory truncation if requested
    T = int(min(T_USE, T))
yw = yw[:, :, :T]
xw = xw[:, :, :T]                 # align state sequence length with observations
T_test = T
M = yw.shape[0]                   # total number of trajectories

# Tolerance Parameter (c) Initialization 
# Ground-truth ambiguity set tolerance, aligned with the truncated trajectory length
c_full = np.asarray(mat["c"]).ravel()
c_true = c_full[:T].astype(np.float64)

# System Model Initialization
sys_model = SystemModel(f, Q, h, R, T, T_test, m, n)
m1x_0 = torch.tensor(mat["x0"], dtype=torch.float32).to(DEVICE)
m2x_0 = torch.tensor(mat["V0"], dtype=torch.float32).to(DEVICE)
sys_model.InitSequence(m1x_0, m2x_0)       # initialize initial state x0 and covariance P0

# Dataset Splitting
# Partition the dataset into Training, Cross-Validation, and Test sets
N_E, N_CV, N_T = int(TRAIN_RATIO * M), int(CV_RATIO * M), int(TEST_RATIO * M)
i0, i1, i2 = N_E, N_E + N_CV, N_E + N_CV + N_T
train_input, train_target = yw[:i0],   xw[:i0]
cv_input,    cv_target    = yw[i0:i1], xw[i0:i1]
test_input,  test_target  = yw[i1:i2], xw[i1:i2]

# Experiment Sizing and Subsampling
# Subsample sequences based on defined experiment sizing parameters, otherwise utilize full splits
train_y = train_input if N_TRAIN_SEQ is None else train_input[:N_TRAIN_SEQ]
train_x = train_target if N_TRAIN_SEQ is None else train_target[:N_TRAIN_SEQ]
cv_y,   cv_x   = cv_input[:N_CV_SEQ], cv_target[:N_CV_SEQ]
test_y, test_x = test_input[:N_TEST], test_target[:N_TEST]

# Setup Summary
print(f"trajectories: {M} total -> {len(train_y)} train / {len(cv_y)} cv / {len(test_y)} test")
print(f"sequence length T = {T}   (state dim m = {m}, obs dim n = {n})")
print(f"true c(t): {c_true[0]:.4f} -> {c_true[-1]:.4f}   "
      f"(saved c has {len(c_full)} samples, first {T} used)")

trajectories: 100 total -> 60 train / 4 cv / 10 test
sequence length T = 1000   (state dim m = 2, obs dim n = 1)
true c(t): 0.1000 -> 0.1000   (saved c has 2000 samples, first 1000 used)


## 4. Data Visualization and State Scale Analysis

To verify the integrity of the data loading and alignment phases, this section visualizes a randomly sampled trajectory from the training subset. The visualization plots the temporal evolution of the latent state vector components ($x_t$) specifically, position ($x_1$) and velocity ($x_2$)—alongside the ground-truth tolerance parameter ($c(t)$) that bounds the ambiguity set.

> **Note on State Scale and Evaluation Metrics:** 
> The cell also computes the maximum absolute amplitudes for both state components across the entire training set. Because the position scale significantly exceeds the velocity scale, the overall Mean-Squared Error (MSE) metric will be heavily dominated by position estimation errors.

In [5]:
# randomly sample a single trajectory index from the training subset
k_show = random.randrange(len(train_y))

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06)
fig.add_trace(go.Scatter(y=train_x[k_show, 0].cpu().numpy(), mode="lines",
                          line=dict(color="#1f77b4", width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(y=train_x[k_show, 1].cpu().numpy(), mode="lines",
                          line=dict(color="#d62728", width=1.2)), row=2, col=1)
fig.add_trace(go.Scatter(y=c_true, mode="lines",
                          line=dict(color="black", width=2.0)), row=3, col=1)

fig.update_yaxes(title_text="x₁ (position)", row=1, col=1)
fig.update_yaxes(title_text="x₂ (velocity)", row=2, col=1)
fig.update_yaxes(title_text="true c(t)", row=3, col=1)
fig.update_xaxes(title_text="t", row=3, col=1)
fig.update_layout(title=f"Training trajectory #{k_show}", height=650, width=750,
                   template="plotly_white", showlegend=False)
fig.show()

# Calculate and output the maximum absolute amplitudes of the state components
# to explicitly highlight the dominant factor in the MSE objective function.
print(f"state scale: |x1| up to {train_x[:, 0].abs().max():.1f}, "
      f"|x2| up to {train_x[:, 1].abs().max():.2f}  "
      "(the MSE is dominated by the position component)")

state scale: |x1| up to 8640.2, |x2| up to 16.27  (the MSE is dominated by the position component)


## 5. Estimators Instantiation

All three estimators are instantiated from the generalized robust Kalman filtering framework. They share a unified operational interface, which allows the training pipeline to drive them uniformly during the evaluation and backpropagation phases:

> ```python
> model.reset_state(y)      # Initializes a new trajectory and resets the recursive hidden states
> model.fnREKF(train=...)   # Executes the robust filtering recursion over the entire time horizon
> model.Xn                  # Yields the posterior state estimate \(\hat{x}_{t|t}\)$\hat{x}_{t\vert{}t}$

In [6]:
models, nets = {}, {}

if INCLUDE_REKF:
    # Analytic REKF: same filter class, fixed scalar tolerance (no network).
    models["REKF"] = Prop_RTKnet(sys_model, train_y[0], 1e-3, use_nn=False, sl_model=0)

if INCLUDE_ORIGINAL:
    # Baseline RT-KalmanNet: Architecture utilizing an MLP
    torch.manual_seed(SEED)                       # reproducible MLP init
    models["Original"] = Orig_RTKnet(sys_model, train_y[0], use_nn=True, **ORIG_MODEL)
    nets["Original"] = models["Original"].nn

if INCLUDE_PROPOSED:
    # Proposed RT-KalmanNet: Upgraded architecture utilizing a GRU
    torch.manual_seed(SEED)                       # reproducible GRU init
    models["Proposed"] = Prop_RTKnet(sys_model, train_y[0], use_nn=True, **PROP_MODEL)
    nets["Proposed"] = models["Proposed"].nn

# Neural Network Module Allocation
for name, net in nets.items():
    net.to(DEVICE)
    n_par = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"{name:9s}: {n_par:,} trainable parameters")

Original : 2,011 trainable parameters
Proposed : 116,865 trainable parameters


## 6. Shared Evaluation and Execution Helpers

This section establishes the unified functional framework required to execute the estimators over individual sequences and evaluate their performance across the designated test set.

The `fnREKF` method variants uniformly return a structured list where the final element indicates the computational elapsed time, and the second element (index `[1]`) provides the estimated tolerance sequence $c(t)$. For the analytic REKF baseline, this tolerance element evaluates to `None`. This standardized interface enables a single wrapper to seamlessly orchestrate all three estimator architectures.

The evaluation protocol strictly adheres to the established benchmark methodology: it generates **one record per test trajectory**, capturing both the Mean-Squared Error (MSE) objective metric and the computational inference time. These metrics are subsequently aggregated for comparative performance analysis.

In [7]:
def run_filter(model, y, train=False):
    """Runs one observation sequence through the filter.

    fnREKF returns a list whose last element is the elapsed time and whose
    element [1] is the estimated tolerance sequence (None for the analytic REKF).
    Original's fnREKF has no bptt_truncation argument, so it is simply not passed
    here -- Proposed's version already defaults to full BPTT (bptt_truncation=None).
    """
    model.reset_state(y)
    out = model.fnREKF(train=train)
    c_arr = out[1]
    c_traj = None if c_arr is None else np.asarray([float(c) for c in c_arr], dtype=np.float64)
    return model.Xn, c_traj, float(out[-1])

def mse_stats_db(v):
    """Mean and spread (dB) of a set of per-sequence linear MSEs."""
    v = np.asarray(v, dtype=np.float64)
    mean_db = to_dB(v.mean())
    spread = to_dB(v.std(ddof=1) + v.mean()) - mean_db if v.size > 1 else 0.0
    return mean_db, spread

## 7. REKF baseline — tolerance grid search

The analytic REKF uses **one constant** tolerance. Following [UAVF] Sec. IV, $c^*$ is selected as the
value minimising the state MSE over a finite grid evaluated on training sequences. The grid is
geometric because the interesting range of $c$ is small.

In [8]:
c_star = None
if INCLUDE_REKF:
    t0 = time.time()
    c_idx = random.sample(range(len(train_y)), min(N_C_SEQ, len(train_y)))
    rekf = models["REKF"]
    grid = {}
    with torch.no_grad():
        for cand in tqdm(C_GRID, desc="REKF grid search", unit="c", leave=False):
            cand = float(cand)
            # For use_nn=False the tolerance is a fixed tensor read by fnComputeTheta.
            rekf.c = torch.tensor(cand, device=DEVICE, dtype=torch.float32)
            losses = [crit(run_filter(rekf, train_y[i])[0], train_x[i]).item() for i in c_idx]
            grid[cand] = float(np.mean(losses))
    c_star = min(grid, key=grid.get)
    rekf.c = torch.tensor(float(c_star), device=DEVICE, dtype=torch.float32)

    for cand, v in grid.items():
        print(f"  c = {cand:8.4g}   train MSE = {to_dB(v):+7.3f} dB"
              f"{'   <-- c*' if cand == c_star else ''}")
    print(f"\nc* = {c_star:.4g}   (true c(t) spans [{c_true.min():.4g}, {c_true.max():.4g}])"
          f"   [{time.time() - t0:.0f}s]")

REKF grid search:   0%|          | 0/20 [00:00<?, ?c/s]

  c =   0.0001   train MSE =  -1.648 dB
  c = 0.0001624   train MSE =  -1.662 dB
  c = 0.0002637   train MSE =  -1.679 dB
  c = 0.0004281   train MSE =  -1.700 dB
  c = 0.0006952   train MSE =  -1.724 dB
  c = 0.001129   train MSE =  -1.755 dB
  c = 0.001833   train MSE =  -1.792 dB
  c = 0.002976   train MSE =  -1.834 dB
  c = 0.004833   train MSE =  -1.883 dB
  c = 0.007848   train MSE =  -1.937 dB
  c =  0.01274   train MSE =  -1.993 dB
  c =  0.02069   train MSE =  -2.049 dB
  c =   0.0336   train MSE =  -2.097 dB
  c =  0.05456   train MSE =  -2.132 dB
  c =  0.08859   train MSE =  -2.144 dB   <-- c*
  c =   0.1438   train MSE =  -2.125 dB
  c =   0.2336   train MSE =  -2.069 dB
  c =   0.3793   train MSE =  -1.973 dB
  c =   0.6158   train MSE =  -1.842 dB
  c =        1   train MSE =  -1.683 dB

c* = 0.08859   (true c(t) spans [0.1, 0.1])   [605s]


## 8. Training through `Pipeline_REKF`

For each network we build the `args` object, instantiate the pipeline and call `NNTrain` — the
pipeline owns the epoch loop, the mini-batching (`DataLoader`, shuffled), the validation pass and the
best-checkpoint saving.

Two pipeline details worth knowing:

* `n_steps` counts **epochs** (the whole training set is swept every epoch).
* `allow_tbptt` must be `True` **only** for the Proposed model: the Original model's `fnREKF()` does
  not accept a `bptt_truncation` argument.

> **Note on the Original network.** Its tolerance reaches the loss only through the `theta`
> bisection, which is a sequence of branches on `c` and therefore not differentiable; its output
> feedback is detached as well. Consequently its parameters receive no gradient and "training" it
> reduces to selecting the best-validation epoch of an essentially unchanged network. This is a
> structural property of the original design — it is exactly the comparison being made, not a bug —
> and it is verified explicitly below.

**Training time is measured here** (wall clock around `NNTrain`) and reported later together with the
inference times.

In [9]:
histories, train_times, ckpts = {}, {}, {}

def train_model(name, train_params, allow_tbptt):
    """Trains one network through Pipeline_REKF and records its wall-clock training time."""
    model = models[name]

    args = config.general_settings()
    args.use_cuda = USE_CUDA
    args.seed = SEED
    args.N_E, args.N_CV, args.N_T = len(train_y), len(cv_y), len(test_y)
    args.T, args.T_test = T, T_test
    for k, v in train_params.items():
        setattr(args, k, v)

    pipeline = Pipeline_REKF(CKPT_DIR, f"RTKnet_{name}", allow_tbptt=allow_tbptt)
    pipeline.setssModel(sys_model)
    pipeline.setModel(model)
    pipeline.setTrainingParams(args)

    print(f"\n##### training {name} RT-KalmanNet #####")
    t0 = time.time()
    cv_lin, cv_dB, tr_lin, tr_dB = pipeline.NNTrain(
        sys_model, cv_y, cv_x, train_y, train_x, PATH_RESULTS)
    elapsed = time.time() - t0

    histories[name] = {"train_dB": tr_dB.tolist(), "val_dB": cv_dB.tolist()}
    train_times[name] = elapsed
    ckpts[name] = pipeline._best_model_path(PATH_RESULTS)
    print(f"  done in {elapsed:.0f}s | best CV = {float(pipeline.MSE_cv_dB_opt):+.4f} dB "
          f"(epoch {pipeline.MSE_cv_idx_opt + 1}) | checkpoint: {ckpts[name]}")
    return pipeline

def load_or_train(name, train_params, allow_tbptt):
    """RETRAIN=True -> train and save; False -> load this shape's checkpoint.

    The checkpoint lives in CKPT_DIR, which is keyed by DATASET_KIND, so the model trained
    on one c(t) shape can never be used to evaluate another.
    Never trains silently: a missing checkpoint in load mode raises FileNotFoundError.
    """
    ckpt = os.path.join(PATH_RESULTS, f"best-model_RTKnet_{name}.pt")
    if RETRAIN:
        return train_model(name, train_params, allow_tbptt)      # trains AND saves
    if not os.path.exists(ckpt):
        raise FileNotFoundError(
            f"No trained model for '{name}' on the '{DATASET_KIND}' dataset at {ckpt}.\n"
            f"Set RETRAIN = True in the configuration cell and re-run to "
            f"create it, then switch back to False.")
    nets[name].load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    ckpts[name] = ckpt
    print(f"[{DATASET_KIND}] loaded {name:9s} from {ckpt}")
    return None

pipelines = {}
if INCLUDE_ORIGINAL:
    pipelines["Original"] = load_or_train("Original", ORIG_TRAIN, allow_tbptt=False)
    if RETRAIN:
        n_flat = sum(1 for p in nets["Original"].parameters()
                     if p.grad is None or p.grad.abs().sum() == 0)
        print(f"  note: {n_flat}/{len(list(nets['Original'].parameters()))} Original tensors "
              f"received no gradient (expected -- see the note above)")

if INCLUDE_PROPOSED:
    pipelines["Proposed"] = load_or_train("Proposed", PROP_TRAIN, allow_tbptt=True)

# ---- training history -----------------------------------------------------------------
# Saved when training, reloaded when loading, so the loss curves in Section 9 and the
# training times in Section 11 survive a restart with RETRAIN = False. HIST_PATH is the
# shape's own file, set in Sec. 2 alongside CKPT_DIR.
if RETRAIN:
    with open(HIST_PATH, "w") as fh:
        json.dump({"histories": histories, "train_times": train_times}, fh, indent=1)
    print(f"saved training history -> {HIST_PATH}")
elif os.path.exists(HIST_PATH):
    with open(HIST_PATH) as fh:
        _hist = json.load(fh)
    histories.update(_hist["histories"])
    train_times.update(_hist["train_times"])
    print(f"loaded training history <- {HIST_PATH}  ({', '.join(histories)})")
else:
    print(f"no training history at {HIST_PATH}\n"
          f"  (run once with RETRAIN = True to create it; the loss curves in\n"
          f"   Section 9 and train_time_s in Section 11 stay empty until then)")

[constant] loaded Original  from Results_task3\constant/best-model_RTKnet_Original.pt
[constant] loaded Proposed  from Results_task3\constant/best-model_RTKnet_Proposed.pt
loaded training history <- Results_task3\constant/training_history.json  (Original, Proposed)


## 9. Training and validation loss

One panel per trained network. A flat curve for the Original network is the expected consequence of
the non-differentiable tolerance path discussed above.

In [10]:
if histories:
    names = list(histories.keys())
    fig = make_subplots(rows=1, cols=len(names),
                         subplot_titles=[f"{n} RT-KalmanNet" for n in names])
    for col, name in enumerate(names, start=1):
        h = histories[name]
        ep = list(range(1, len(h["train_dB"]) + 1))
        fig.add_trace(go.Scatter(x=ep, y=h["train_dB"], mode="lines", name="train",
                                  line=dict(color="#1f77b4", width=2), legendgroup="train",
                                  showlegend=(col == 1)), row=1, col=col)
        fig.add_trace(go.Scatter(x=ep, y=h["val_dB"], mode="lines", name="validation",
                                  line=dict(color="#ff7f0e", width=2, dash="dash"),
                                  legendgroup="val", showlegend=(col == 1)), row=1, col=col)
        fig.update_xaxes(title_text="epoch", row=1, col=col)
    fig.update_yaxes(title_text="MSE [dB]", row=1, col=1)
    fig.update_layout(title="Training and validation loss", height=450, width=550 * len(names),
                       template="plotly_white", hovermode="x unified")
    fig.show()
else:
    print("no network was trained")

## 10. Test — state MSE and computation time

Every estimator is run over the same test trajectories. For each trajectory we record

* the **posterior state MSE** (the shared comparison metric), and
* the **computation time** of the REKF recursion, as returned by `fnREKF`,

which is exactly the per-sequence bookkeeping used in the professor's notebook. The best-validation
checkpoint is reloaded before testing so that the reported numbers correspond to the selected model.

In [11]:
# Reload the best-validation weights (the pipeline saved them during training).
for name, path in ckpts.items():
    if os.path.exists(path):
        nets[name].load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
        print(f"{name}: loaded {path}")

per_seq, c_est = {}, {}
print("\n##### evaluating MSE and computation time on the test sequences #####")
for name in [k for k in ("REKF", "Original", "Proposed") if k in models]:
    model = models[name]
    net = getattr(model, "nn", None)
    if net is not None:
        net.eval()

    t0 = time.time()
    rows, c_trajs = [], []
    with torch.no_grad():
        for k_seq in tqdm(range(len(test_y)), desc=f"test {name}", unit="traj", leave=False):
            Xn, c_traj, elapsed = run_filter(model, test_y[k_seq])
            rows.append({"model": name, "sequence": k_seq,
                         "mse": crit(Xn, test_x[k_seq]).item(),
                         "comp_time_s": elapsed})
            if c_traj is not None:
                c_trajs.append(c_traj)

    df = pd.DataFrame(rows)
    per_seq[name] = df
    if c_trajs:
        c_est[name] = np.stack(c_trajs)            # [n_test, T]

    mean_db, spread_db = mse_stats_db(df["mse"].values)
    print(f"  {name:9s}: MSE = {mean_db:+7.3f} dB (spread {spread_db:.3f}) | "
          f"comp. time = {df['comp_time_s'].mean():.3f} s/traj  [{time.time() - t0:.0f}s]")

per_seq_df = pd.concat(per_seq.values(), ignore_index=True) if per_seq else pd.DataFrame()

Original: loaded Results_task3\constant/best-model_RTKnet_Original.pt
Proposed: loaded Results_task3\constant/best-model_RTKnet_Proposed.pt

##### evaluating MSE and computation time on the test sequences #####


test REKF:   0%|          | 0/10 [00:00<?, ?traj/s]

  REKF     : MSE =  -1.949 dB (spread 0.368) | comp. time = 5.901 s/traj  [59s]


test Original:   0%|          | 0/10 [00:00<?, ?traj/s]

  Original : MSE =  -1.780 dB (spread 0.311) | comp. time = 7.028 s/traj  [70s]


test Proposed:   0%|          | 0/10 [00:00<?, ?traj/s]

  Proposed : MSE =  -1.953 dB (spread 0.374) | comp. time = 7.394 s/traj  [74s]


## 11. Summary table

The headline comparison, extended with the timing columns. `c_mean` is the average estimated
tolerance — a constant $c^*$ for the REKF and the mean of $\hat c(t)$ for the two networks.

In [12]:
rows = []
for name, df in per_seq.items():
    mean_db, spread_db = mse_stats_db(df["mse"].values)
    if name == "REKF":
        note = f"c* = {c_star:.4g}"
    else:
        note = f"mean c_t = {c_est[name].mean():.4g}" if name in c_est else ""
    rows.append({
        "model": name,
        "mse_db": round(mean_db, 3),
        "std_db": round(spread_db, 3),
        "train_time_s": round(train_times.get(name, float("nan")), 1),
        "test_time_s": round(df["comp_time_s"].sum(), 2),
        "time_per_traj_s": round(df["comp_time_s"].mean(), 3),
        "time_per_step_ms": round(1e3 * df["comp_time_s"].mean() / T, 3),
        "note": note,
    })

results = pd.DataFrame(rows).sort_values("mse_db").reset_index(drop=True)
results.to_csv(os.path.join(CKPT_DIR, "results_task3_new.csv"), index=False)
per_seq_df.to_csv(os.path.join(CKPT_DIR, "per_sequence_metrics.csv"), index=False)

print(f"posterior state MSE over {len(test_y)} test trajectories (T = {T})\n")
# print(results.to_string(index=False))
results

posterior state MSE over 10 test trajectories (T = 1000)



,model,mse_db,std_db,train_time_s,test_time_s,time_per_traj_s,time_per_step_ms,note
0,Proposed,-1.953,0.374,6041.9,73.94,7.394,7.394,mean c_t = 0.07405
1,REKF,-1.949,0.368,NaN,59.01,5.901,5.901,c* = 0.08859
2,Original,-1.780,0.311,869.7,70.28,7.028,7.028,mean c_t = 0.4844


## 12. Computational Complexity and Inference Time Metrics

This section evaluates the computational efficiency of the implemented estimators, adhering to the benchmarking methodology established for this project. Because all evaluated models execute the identical underlying robust filtering recursion, any variations in inference time strictly isolate the computational overhead introduced by the tolerance $c$ estimation mechanism. 

We compare the analytic REKF (which utilizes a fixed scalar tolerance and operates without training) against the neural-network-aided architectures: the Original RT-KalmanNet (MLP-based) and the Proposed RT-KalmanNet (GRU-based). The reported quantitative metrics include:

*   **Training Time:** The total wall-clock duration of the backpropagation and optimization phase (zero for the analytic REKF).
*   **Total Test Time:** The cumulative computational time required to filter the entire test dataset.
*   **Normalized Inference Time:** The execution time evaluated per trajectory and per individual time step. These scale-free metrics are crucial for demonstrating the real-time applicability of the neural-augmented estimators compared to traditional, computationally heavy model-based approaches.

In [13]:
if not per_seq_df.empty:
    order = [k for k in ("REKF", "Original", "Proposed") if k in per_seq]
    colors = {"REKF": "#7f7f7f", "Original": "#ff7f0e", "Proposed": "#1f77b4"}

    fig = make_subplots(rows=1, cols=3, subplot_titles=(
        "Inference time per trajectory<br>(dashed = mean)",
        "State MSE per trajectory",
        f"Mean inference time<br>(T = {T} steps)"))

    # per-sequence computation time + mean
    for name in order:
        d = per_seq[name]
        fig.add_trace(go.Scatter(x=d["sequence"], y=d["comp_time_s"], mode="lines+markers",
                                  name=name, line=dict(color=colors[name], width=1.2),
                                  marker=dict(size=5)), row=1, col=1)
        fig.add_hline(y=d["comp_time_s"].mean(), line=dict(color=colors[name], dash="dash", width=1),
                      opacity=0.6, row=1, col=1)

    # per-sequence MSE in dB
    for name in order:
        d = per_seq[name]
        fig.add_trace(go.Scatter(x=d["sequence"], y=10 * np.log10(d["mse"]), mode="lines+markers",
                                  name=name, line=dict(color=colors[name], width=1.2),
                                  marker=dict(size=5), showlegend=False), row=1, col=2)

    # mean time per trajectory (bar) with std error bars
    means = [per_seq[k]["comp_time_s"].mean() for k in order]
    stds = [per_seq[k]["comp_time_s"].std(ddof=1) if len(per_seq[k]) > 1 else 0.0 for k in order]
    fig.add_trace(go.Bar(x=order, y=means, error_y=dict(type="data", array=stds, visible=True),
                          marker_color=[colors[k] for k in order], showlegend=False), row=1, col=3)

    fig.update_xaxes(title_text="test sequence", row=1, col=1)
    fig.update_yaxes(title_text="computation time [s]", row=1, col=1)
    fig.update_xaxes(title_text="test sequence", row=1, col=2)
    fig.update_yaxes(title_text="MSE [dB]", row=1, col=2)
    fig.update_yaxes(title_text="time per trajectory [s]", row=1, col=3)

    fig.update_layout(title="Computation-time metrics", height=450, width=1500,
                       template="plotly_white")
    fig.show()

    print("training time [s]:", {k: round(v, 1) for k, v in train_times.items()})

training time [s]: {'Original': 869.7, 'Proposed': 6041.9}


## 13. Estimated tolerance $\hat c(t)$ vs ground truth

**How each method represents $c$** — checked in the source rather than assumed:

* **Proposed** (`RTKnet/Proposed/RT_KalmanNet_nn.py`) — a GRU followed by `sigmoid`, producing a
  **continuous** $\hat c_t \in (0,1)$ at every step. Its output bias is initialised at $0$, so the
  filter starts at $c = \sigma(0) = 0.5$ — the same starting point as the Original network, so that
  the comparison between the two architectures is not biased by the initialisation.
* **Original** (`RTKnet/Original/RT_KalmanNet_nn.py`) — an MLP with (detached) output feedback, also
  followed by `sigmoid`, so it too produces a **continuous** $\hat c_t \in (0,1)$ at every step
  (`c_array` in `fnREKF`).
* **REKF** — a single **constant** $c^*$, drawn as a horizontal reference line.

Both networks therefore expose the *same kind of object* — a per-step continuous trajectory of the
same length — so a direct, like-for-like comparison against $c_{\text{true}}(t)$ is legitimate and no
artificial mapping is needed. The only asymmetry to keep in mind when reading the figures is that the
Original network's tolerance is not trainable (Section 8), so its curve reflects an untrained network.

The band shows the spread across test trajectories (the networks see different measurement
realisations, so $\hat c(t)$ is trajectory-dependent, whereas $c_{\text{true}}(t)$ is common to all).

In [14]:
steps = np.arange(1, T + 1)
band_rgba = {"Original": "rgba(255,127,14,0.18)", "Proposed": "rgba(31,119,180,0.18)"}

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=c_true, mode="lines", name="true c(t) (from LDF.m)",
                          line=dict(color="black", width=2.4)))

if c_star is not None:
    fig.add_hline(y=c_star, line=dict(color="#7f7f7f", dash="dash", width=1.6),
                  annotation_text=f"REKF constant c* = {c_star:.4g}",
                  annotation_position="top left")

for name, color in (("Original", "#ff7f0e"), ("Proposed", "#1f77b4")):
    if name not in c_est:
        continue
    Ce = c_est[name]                       # [n_test, T]
    mean, std = Ce.mean(axis=0), Ce.std(axis=0)
    fig.add_trace(go.Scatter(x=np.concatenate([steps, steps[::-1]]),
                              y=np.concatenate([mean + std, (mean - std)[::-1]]),
                              fill="toself", fillcolor=band_rgba[name],
                              line=dict(width=0), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=steps, y=mean, mode="lines",
                              name=f"{name} RT-KalmanNet (mean)", line=dict(color=color, width=1.6)))

fig.update_layout(
    title=f"Estimated vs true tolerance — mean over {len(test_y)} test trajectories "
          "(band = ±1 std)",
    xaxis_title="t", yaxis_title="tolerance c",
    template="plotly_white", hovermode="x unified", height=480, width=1000)
fig.show()

## 14. One test trajectory in detail

The aggregated metrics above summarise performance across the whole test set; this section zooms
into a single, randomly-picked trajectory to see what that behaviour looks like concretely, on the
best-performing network (chosen automatically from the summary table).

Five panels, top to bottom:

* **position** and **velocity** — true state vs. estimate;
* the **observation** actually fed to the filter;
* the **tolerance** $c(t)$ on this specific trajectory — true value, REKF's constant $c^*$, and each
  network's estimate;
* the **squared tolerance error** $(\hat c(t) - c_{\text{true}}(t))^2$, log scale — the
  single-trajectory analogue of the aggregated $\mathrm{MSE}(t)$ from Section 14.

A single realisation is noisier than an average, so expect more jitter here than in the aggregated
figures — that is expected, not a sign of a problem.

In [15]:
k = random.randrange(len(test_y))
best = "Proposed" if "Proposed" in models else ("Original" if "Original" in models else "REKF")

with torch.no_grad():
    Xn_k, c_k, _ = run_filter(models[best], test_y[k])

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                     subplot_titles=("x₁ (position)", "x₂ (velocity)",
                                      "y₁ (observation)", "tolerance c",
                                      "(ĉ − c)²"))

# position: true vs. estimated
fig.add_trace(go.Scatter(y=test_x[k, 0].cpu().numpy(), mode="lines", name="real",
                          line=dict(color="blue", width=1.2), legendgroup="real"), row=1, col=1)
fig.add_trace(go.Scatter(y=Xn_k[0].cpu().numpy(), mode="lines", name=f"estimated ({best})",
                          line=dict(color="blue", width=1.0, dash="dash"), legendgroup="est"),
              row=1, col=1)

# velocity: true vs. estimated
fig.add_trace(go.Scatter(y=test_x[k, 1].cpu().numpy(), mode="lines", name="real",
                          line=dict(color="red", width=1.2), legendgroup="real", showlegend=False),
              row=2, col=1)
fig.add_trace(go.Scatter(y=Xn_k[1].cpu().numpy(), mode="lines", name=f"estimated ({best})",
                          line=dict(color="red", width=1.0, dash="dash"), legendgroup="est",
                          showlegend=False), row=2, col=1)

# the observation the filter actually saw
fig.add_trace(go.Scatter(y=test_y[k, 0].cpu().numpy(), mode="lines",
                          line=dict(color="green", width=0.9), showlegend=False), row=3, col=1)

# tolerance on this trajectory: true c(t), REKF's constant c*, each network's c_hat(t)
fig.add_trace(go.Scatter(x=steps, y=c_true, mode="lines", name="true c(t)",
                          line=dict(color="black", width=2.0)), row=4, col=1)
for name, color in (("Original", "#ff7f0e"), ("Proposed", "#1f77b4")):
    if name in c_est:
        fig.add_trace(go.Scatter(x=steps, y=c_est[name][k], mode="lines", name=name,
                                  line=dict(color=color, width=1.2, dash="solid")), row=4, col=1)
if c_star is not None:
    fig.add_hline(y=c_star, line=dict(color="#7f7f7f", dash="dot", width=1.4),
                  annotation_text="REKF c*", row=4, col=1)

# squared tolerance error, single-trajectory analogue of Section 14's MSE(t)
for name, color in (("Original", "#ff7f0e"), ("Proposed", "#1f77b4")):
    if name in c_est:
        fig.add_trace(go.Scatter(x=steps, y=(c_est[name][k] - c_true) ** 2, mode="lines", name=name,
                                  line=dict(color=color, width=0.9), showlegend=False), row=5, col=1)

fig.update_yaxes(type="log", row=5, col=1)
fig.update_xaxes(title_text="t", row=5, col=1)
fig.update_layout(title=f"Test trajectory #{k}", height=1150, width=850,
                   template="plotly_white", hovermode="x unified")
fig.show()

print(f"\nall checkpoints and tables saved in {CKPT_DIR}/")


all checkpoints and tables saved in Results_task3\constant/
